In [12]:

import random
random.seed(7)


class Environment:
    def __init__(self, size=5):
        self.size = size
        self.dirt = [random.choice([True, False]) for _ in range(size)]
        self.agent_position = 0

    def get_percept(self):
        return {
            "position": self.agent_position,
            "is_dirty": self.dirt[self.agent_position],
        }

    def apply_action(self, action):
        if action == "Suck":
            self.dirt[self.agent_position] = False

        elif action == "MoveRight" and self.agent_position < self.size - 1:
            self.agent_position += 1

        elif action == "MoveLeft" and self.agent_position > 0:
            self.agent_position -= 1

    def is_clean(self):
        return not any(self.dirt)

    def render(self):
        row = ""
        for i in range(self.size):
            cell = "D" if self.dirt[i] else "."
            if i == self.agent_position:
                cell = "[" + cell + "]"
            row += cell
        return row


In [5]:

def simple_reflex_agent(percept, size):
    if percept["is_dirty"]:
        return "Suck"
    elif percept["position"] < size - 1:
        return "MoveRight"
    else:
        return "NoOp"


In [6]:

env = Environment(size=5)
print("Start:", env.render())

for step in range(10):
    percept = env.get_percept()
    action = simple_reflex_agent(percept, env.size)

    print(f"Step {step+1}: percept={percept}, action={action}")

    env.apply_action(action)
    print("   World:", env.render())

    if env.is_clean() and percept["position"] == env.size - 1:
        print("All clean, agent stops.")
        break


Start: [.]D.DD
Step 1: percept={'position': 0, 'is_dirty': False}, action=MoveRight
   World: .[D].DD
Step 2: percept={'position': 1, 'is_dirty': True}, action=Suck
   World: .[.].DD
Step 3: percept={'position': 1, 'is_dirty': False}, action=MoveRight
   World: ..[.]DD
Step 4: percept={'position': 2, 'is_dirty': False}, action=MoveRight
   World: ...[D]D
Step 5: percept={'position': 3, 'is_dirty': True}, action=Suck
   World: ...[.]D
Step 6: percept={'position': 3, 'is_dirty': False}, action=MoveRight
   World: ....[D]
Step 7: percept={'position': 4, 'is_dirty': True}, action=Suck
   World: ....[.]
All clean, agent stops.


In [7]:
#Lab task 1

import random

class GridEnvironment:
    def __init__(self, rows, cols, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)

        self.rows, self.cols = rows, cols

        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]

        self.agent_pos = (0, 0)

    def get_percept(self):
        r, c = self.agent_pos
        return {
            "position": (r, c),
            "is_dirty": self.dirt[r][c]
        }

    def valid_moves(self):
        r, c = self.agent_pos
        moves = []

        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")

        return moves

    def apply_action(self, action):
        r, c = self.agent_pos

        if action == "Suck":
            self.dirt[r][c] = False

        elif action == "Right" and c < self.cols - 1:
            self.agent_pos = (r, c + 1)

        elif action == "Left" and c > 0:
            self.agent_pos = (r, c - 1)

        elif action == "Down" and r < self.rows - 1:
            self.agent_pos = (r + 1, c)

        elif action == "Up" and r > 0:
            self.agent_pos = (r - 1, c)

    def is_clean(self):
        return all(
            not any(row) for row in self.dirt
        )

    def render(self):
        for r in range(self.rows):
            line = ""
            for c in range(self.cols):
                cell = "D" if self.dirt[r][c] else "."
                if (r, c) == self.agent_pos:
                    cell = "[" + cell + "]"
                line += cell + " "
            print(line)
        print()


# DECIDE (THE AGENT)
# Clean current room if dirty, otherwise move randomly.

def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"

    return random.choice(env.valid_moves())


# RUN THE LOOP

def run_simulation(env, max_steps=200):
    steps = 0

    for _ in range(max_steps):
        percept = env.get_percept()

        if env.is_clean():
            break

        action = simple_reflex_agent(percept, env)

        env.apply_action(action)
        steps += 1

    return steps


# MAIN PROGRAM

env = GridEnvironment(
    rows=4,
    cols=4,
    dirt_probability=0.5,
    seed=43
)

print("INITIAL GRID:")
env.render()

steps = run_simulation(env, max_steps=200)

print("FINAL GRID:")
env.render()

print("Steps:", steps)
print("All Clean:", env.is_clean())


INITIAL GRID:
[D] . D D 
. . D D 
D D D . 
. . D D 

FINAL GRID:
. . . . 
. . . . 
[.] . . . 
. . . . 

Steps: 112
All Clean: True


In [9]:
#Lab task 2

import random
import copy

# ==========================================
# 1. GRID ENVIRONMENT
# ==========================================

class GridEnvironment:
    def __init__(self, rows, cols, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)

        self.rows, self.cols = rows, cols

        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]

        self.agent_pos = (0, 0)

    def get_percept(self):
        r, c = self.agent_pos
        return {
            "position": (r, c),
            "is_dirty": self.dirt[r][c]
        }

    def valid_moves(self):
        r, c = self.agent_pos
        moves = []

        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")

        return moves

    def apply_action(self, action):
        r, c = self.agent_pos

        if action == "Suck":
            self.dirt[r][c] = False

        elif action == "Right" and c < self.cols - 1:
            self.agent_pos = (r, c + 1)

        elif action == "Left" and c > 0:
            self.agent_pos = (r, c - 1)

        elif action == "Down" and r < self.rows - 1:
            self.agent_pos = (r + 1, c)

        elif action == "Up" and r > 0:
            self.agent_pos = (r - 1, c)

    def is_clean(self):
        return all(not any(row) for row in self.dirt)

    def render(self):
        for r in range(self.rows):
            line = ""
            for c in range(self.cols):
                cell = "D" if self.dirt[r][c] else "."
                if (r, c) == self.agent_pos:
                    cell = "[" + cell + "]"
                line += cell + " "
            print(line)
        print()


# ==========================================
# 2. SIMPLE REFLEX AGENT
# ==========================================

def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"

    return random.choice(env.valid_moves())


# ==========================================
# 3. MODEL-BASED REFLEX AGENT
# ==========================================

def neighbor_position(pos, action):
    r, c = pos

    return {
        "Right": (r, c + 1),
        "Left": (r, c - 1),
        "Down": (r + 1, c),
        "Up": (r - 1, c)
    }[action]


class ModelBasedAgent:
    def __init__(self):
        self.visited = set()

    def act(self, percept, env):
        self.visited.add(percept["position"])

        if percept["is_dirty"]:
            return "Suck"

        moves = env.valid_moves()
        here = percept["position"]

        unvisited = [
            m for m in moves
            if neighbor_position(here, m) not in self.visited
        ]

        if unvisited:
            return random.choice(unvisited)

        return random.choice(moves)


# ==========================================
# 4. RUN SIMULATION
# ==========================================

def run_agent(env, agent_function, max_steps=200):
    steps = 0

    for _ in range(max_steps):
        if env.is_clean():
            break

        percept = env.get_percept()
        action = agent_function(percept, env)

        env.apply_action(action)
        steps += 1

    return steps


# ==========================================
# 5. RUN AND COMPARE BOTH AGENTS
# ==========================================

random.seed(11)

env1 = GridEnvironment(
    rows=4,
    cols=4,
    dirt_probability=0.5
)

dirt_copy = copy.deepcopy(env1.dirt)

print("INITIAL GRID:")
env1.render()

# Simple Reflex Agent
steps1 = run_agent(env1, simple_reflex_agent)

print("SIMPLE REFLEX AGENT:")
print("Steps:", steps1)
print("All Clean:", env1.is_clean())
env1.render()


# Model-Based Agent
random.seed(11)

env2 = GridEnvironment(
    rows=4,
    cols=4,
    dirt_probability=0.5
)

env2.dirt = copy.deepcopy(dirt_copy)

agent2 = ModelBasedAgent()

steps2 = run_agent(env2, agent2.act)

print("MODEL-BASED AGENT:")
print("Steps:", steps2)
print("All Clean:", env2.is_clean())
env2.render()


# ==========================================
# 6. FINAL COMPARISON
# ==========================================

print("========== COMPARISON ==========")
print("Simple Reflex (Random) Steps:", steps1)
print("Model-Based Steps:", steps2)

if steps2 < steps1:
    print("Model-Based Agent performed better!")

elif steps1 < steps2:
    print("Simple Reflex Agent performed better!")

else:
    print("Both agents took equal steps.")


INITIAL GRID:
[D] . . D 
. . D . 
. . D D 
D . . D 

SIMPLE REFLEX AGENT:
Steps: 48
All Clean: True
. . . . 
. . . . 
. . . . 
[.] . . . 

MODEL-BASED AGENT:
Steps: 42
All Clean: True
. . . . 
. . . . 
. . . . 
[.] . . . 

========== COMPARISON ==========
Simple Reflex (Random) Steps: 48
Model-Based Steps: 42
Model-Based Agent performed better!


In [10]:
#Lab task 3

# ==========================================
# LAB TASK 3 - GOAL-BASED AGENT
# ==========================================

class GridEnvironment:
    def __init__(self, rows=4, cols=4):
        self.rows = rows
        self.cols = cols
        self.agent_pos = (0, 0)

    def apply_action(self, action):
        r, c = self.agent_pos

        if action == "Right" and c < self.cols - 1:
            self.agent_pos = (r, c + 1)

        elif action == "Left" and c > 0:
            self.agent_pos = (r, c - 1)

        elif action == "Down" and r < self.rows - 1:
            self.agent_pos = (r + 1, c)

        elif action == "Up" and r > 0:
            self.agent_pos = (r - 1, c)


# ==========================================
# GOAL-BASED AGENT
# ==========================================

def goal_based_agent(position, goal):
    r, c = position
    gr, gc = goal

    if r < gr:
        return "Down"

    if r > gr:
        return "Up"

    if c < gc:
        return "Right"

    if c > gc:
        return "Left"

    return "NoOp"


# ==========================================
# RUN THE LOOP
# ==========================================

env = GridEnvironment(rows=4, cols=4)

env.agent_pos = (0, 0)

goal = (3, 2)

print(f"Start position: {env.agent_pos}, Goal: {goal}")

for step in range(20):

    if env.agent_pos == goal:
        print(f"Reached the goal in {step} steps.")
        break

    action = goal_based_agent(env.agent_pos, goal)

    env.apply_action(action)

    print(
        f"Step {step+1}: {action} -> now at {env.agent_pos}"
    )


# ==========================================
# FINAL RESULT
# ==========================================

print("\nFinal Position:", env.agent_pos)
print("Goal Position:", goal)

if env.agent_pos == goal:
    print("SUCCESS! Agent reached the goal.")
else:
    print("Agent did not reach the goal.")


Start position: (0, 0), Goal: (3, 2)
Step 1: Down -> now at (1, 0)
Step 2: Down -> now at (2, 0)
Step 3: Down -> now at (3, 0)
Step 4: Right -> now at (3, 1)
Step 5: Right -> now at (3, 2)
Reached the goal in 5 steps.

Final Position: (3, 2)
Goal Position: (3, 2)
SUCCESS! Agent reached the goal.


In [4]:

import random
import statistics

# ==========================================
# LAB TASK 5 - FAIR AGENT COMPARISON
# ==========================================

class GridEnvironment:
    def __init__(self, rows=4, cols=4,
                 dirt_probability=0.5, seed=None):

        rng = random.Random(seed)

        self.rows = rows
        self.cols = cols

        self.dirt = [
            [rng.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]

        self.agent_pos = (0, 0)

    def get_percept(self):
        r, c = self.agent_pos
        return {
            "position": (r, c),
            "is_dirty": self.dirt[r][c]
        }

    def valid_moves(self):
        r, c = self.agent_pos
        moves = []

        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")

        return moves

    def apply_action(self, action):
        r, c = self.agent_pos

        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.agent_pos = (r, c + 1)
        elif action == "Left" and c > 0:
            self.agent_pos = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.agent_pos = (r + 1, c)
        elif action == "Up" and r > 0:
            self.agent_pos = (r - 1, c)

    def is_clean(self):
        return all(not any(row) for row in self.dirt)


# ==========================================
# SIMPLE REFLEX AGENT
# ==========================================

def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"

    return random.choice(env.valid_moves())


# ==========================================
# MODEL-BASED AGENT
# ==========================================

def neighbor_position(pos, action):
    r, c = pos
    return {
        "Right": (r, c + 1),
        "Left": (r, c - 1),
        "Down": (r + 1, c),
        "Up": (r - 1, c)
    }[action]


class ModelBasedAgent:
    def __init__(self):
        self.visited = set()

    def act(self, percept, env):
        self.visited.add(percept["position"])

        if percept["is_dirty"]:
            return "Suck"

        moves = env.valid_moves()
        here = percept["position"]

        unvisited = [
            m for m in moves
            if neighbor_position(here, m)
            not in self.visited
        ]

        if unvisited:
            return random.choice(unvisited)

        return random.choice(moves)


# ==========================================
# RUN AGENT
# ==========================================

def run_agent(env, decide, max_steps=200):
    steps = 0

    for _ in range(max_steps):
        if env.is_clean():
            break

        percept = env.get_percept()
        action = decide(percept, env)

        env.apply_action(action)
        steps += 1

    return steps


# ==========================================
# COMPARE ON MANY LAYOUTS
# ==========================================

def compare_agents(num_layouts=5):
    simple_steps = []
    model_steps = []

    print("Layout | Simple Reflex | Model-Based")
    print("------------------------------------")

    for i in range(num_layouts):

        # Same layout for both agents
        env1 = GridEnvironment(
            rows=4, cols=4,
            dirt_probability=0.5,
            seed=i
        )

        env2 = GridEnvironment(
            rows=4, cols=4,
            dirt_probability=0.5,
            seed=i
        )

        # Simple Reflex Agent
        random.seed(1000 + i)
        steps1 = run_agent(
            env1, simple_reflex_agent
        )

        # Model-Based Agent with fresh memory
        agent2 = ModelBasedAgent()

        random.seed(2000 + i)
        steps2 = run_agent(
            env2, agent2.act
        )

        simple_steps.append(steps1)
        model_steps.append(steps2)

        print(f"{i+1:^6} | {steps1:^13} | {steps2:^11}")

    print("\n========== FINAL RESULTS ==========")

    avg_simple = statistics.mean(simple_steps)
    avg_model = statistics.mean(model_steps)

    print("Simple Reflex Average Steps:", round(avg_simple, 2))
    print("Model-Based Average Steps:", round(avg_model, 2))

    if avg_model < avg_simple:
        print("Model-Based Agent performed better!")
    elif avg_simple < avg_model:
        print("Simple Reflex Agent performed better!")
    else:
        print("Both agents performed equally.")


# ==========================================
# MAIN PROGRAM
# ==========================================

compare_agents(num_layouts=20)


Layout | Simple Reflex | Model-Based
------------------------------------
  1    |      77       |     19     
  2    |      108      |     38     
  3    |      76       |     35     
  4    |      59       |     41     
  5    |      110      |     30     
  6    |      72       |     40     
  7    |      105      |     42     
  8    |      95       |     34     
  9    |      54       |     30     
  10   |      51       |     26     
  11   |      43       |     22     
  12   |      48       |     71     
  13   |      95       |     23     
  14   |      34       |     22     
  15   |      70       |     35     
  16   |      55       |     22     
  17   |      46       |     26     
  18   |      42       |     20     
  19   |      91       |     145    
  20   |      53       |     55     

========== FINAL RESULTS ==========
Simple Reflex Average Steps: 69.2
Model-Based Average Steps: 38.8
Model-Based Agent performed better!


In [5]:

# ==========================================
# LAB TASK 6 - UPGRADE THE SENSORS
# ==========================================

# NEW ENVIRONMENT WITH BETTER SENSOR

class SensorEnvironment(GridEnvironment):

    def get_percept(self):
        r, c = self.agent_pos

        neighbours = {}

        for move in self.valid_moves():
            nr, nc = neighbor_position(self.agent_pos, move)

            neighbours[move] = self.dirt[nr][nc]

        return {
            "position": self.agent_pos,
            "is_dirty": self.dirt[r][c],
            "neighbours": neighbours
        }


# ==========================================
# SENSOR-BASED AGENT
# ==========================================

def sensor_agent(percept, env):

    # Clean current room if dirty
    if percept["is_dirty"]:
        return "Suck"

    # Find dirty neighbouring rooms
    dirty_moves = [
        move
        for move, dirty in percept["neighbours"].items()
        if dirty
    ]

    # Move towards a dirty neighbour
    if dirty_moves:
        return random.choice(dirty_moves)

    # If no neighbour is dirty, move randomly
    return random.choice(env.valid_moves())


# ==========================================
# TEST THE NEW SENSOR
# ==========================================

env = SensorEnvironment(
    rows=4,
    cols=4,
    dirt_probability=0.5,
    seed=11
)

print("Current Position:", env.agent_pos)
print("Sensor Percept:", env.get_percept())


# ==========================================
# COMPARE AGENTS ON 20 SAME LAYOUTS
# ==========================================

simple_steps = []
sensor_steps = []

print("\nLayout | Simple Reflex | Sensor Agent")
print("------------------------------------")

for i in range(20):

    # Create the same layout for both agents
    env1 = GridEnvironment(
        rows=4,
        cols=4,
        dirt_probability=0.5,
        seed=i
    )

    env2 = SensorEnvironment(
        rows=4,
        cols=4,
        dirt_probability=0.5,
        seed=i
    )

    random.seed(1000 + i)
    steps1 = run_agent(env1, simple_reflex_agent)

    random.seed(2000 + i)
    steps2 = run_agent(env2, sensor_agent)

    simple_steps.append(steps1)
    sensor_steps.append(steps2)

    print(f"{i+1:^6} | {steps1:^13} | {steps2:^12}")


# ==========================================
# FINAL RESULTS
# ==========================================

avg_simple = statistics.mean(simple_steps)
avg_sensor = statistics.mean(sensor_steps)

print("\n========== FINAL RESULTS ==========")
print("Simple Reflex Average:", round(avg_simple, 2))
print("Sensor Agent Average:", round(avg_sensor, 2))

if avg_sensor < avg_simple:
    print("Sensor Agent performed better!")
elif avg_simple < avg_sensor:
    print("Simple Reflex Agent performed better!")
else:
    print("Both agents performed equally.")


Current Position: (0, 0)
Sensor Percept: {'position': (0, 0), 'is_dirty': True, 'neighbours': {'Right': False, 'Down': False}}

Layout | Simple Reflex | Sensor Agent
------------------------------------
  1    |      77       |      21     
  2    |      108      |      39     
  3    |      76       |      25     
  4    |      59       |      22     
  5    |      110      |      58     
  6    |      72       |      16     
  7    |      105      |      42     
  8    |      95       |      43     
  9    |      54       |      38     
  10   |      51       |      21     
  11   |      43       |      21     
  12   |      48       |      40     
  13   |      95       |      47     
  14   |      34       |      22     
  15   |      70       |      28     
  16   |      55       |      26     
  17   |      46       |      44     
  18   |      42       |      27     
  19   |      91       |      30     
  20   |      53       |      23     

========== FINAL RESULTS ==========


In [14]:
#Task07
import random

def make_grid(seed):
    random.seed(seed)
    dirt = [[random.choice([True, False]) for _ in range(4)]
            for _ in range(4)]
    values = [[random.randint(1, 5) for _ in range(4)]
              for _ in range(4)]
    return dirt, values

def run_agent(seed, utility=False):
    dirt, values = make_grid(seed)
    pos = (0, 0)
    score = 0
    steps = 0

    while steps < 20:
        rooms = [(r, c) for r in range(4) for c in range(4)
                 if dirt[r][c]]

        if not rooms:
            break

        if utility:
            target = max(rooms, key=lambda p:
                values[p[0]][p[1]] /
                (abs(pos[0]-p[0]) + abs(pos[1]-p[1]) + 1))
        else:
            target = min(rooms, key=lambda p:
                abs(pos[0]-p[0]) + abs(pos[1]-p[1]))

        while pos != target and steps < 20:
            r, c = pos
            tr, tc = target

            if r < tr:
                pos = (r+1, c)
            elif r > tr:
                pos = (r-1, c)
            elif c < tc:
                pos = (r, c+1)
            else:
                pos = (r, c-1)

            steps += 1

        if steps < 20:
            r, c = pos
            score += values[r][c]
            dirt[r][c] = False
            steps += 1

    return score

nearest_scores = [run_agent(i, False) for i in range(200)]
utility_scores = [run_agent(i, True) for i in range(200)]

print("Nearest-room average:", round(sum(nearest_scores)/200, 2))
print("Utility average:", round(sum(utility_scores)/200, 2))
print("Utility wins:", sum(
    utility_scores[i] > nearest_scores[i] for i in range(200)
), "out of 200")


Nearest-room average: 22.92
Utility average: 22.5
Utility wins: 29 out of 200


In [18]:

# TASK 8: GUARDRAILS

restricted_rooms = {(1, 1), (2, 2)}
approval_rooms = {(0, 2), (3, 1)}
action_log = []

def ask_human(position):
    return False  # Human refuses approval


def guarded_step(env, action):
    old_position = env.agent_pos
    r, c = old_position
    status = "Allowed"

    if action in ["Up", "Down", "Left", "Right"]:
        new_position = {
            "Up": (r - 1, c),
            "Down": (r + 1, c),
            "Left": (r, c - 1),
            "Right": (r, c + 1)
        }[action]

        if (new_position[0] < 0 or new_position[0] >= env.rows
                or new_position[1] < 0 or new_position[1] >= env.cols):
            status = "Blocked: outside grid"

        elif new_position in restricted_rooms:
            status = "Blocked: restricted room"

        else:
            env.apply_action(action)

    elif action == "Suck":
        if old_position in restricted_rooms:
            status = "Blocked: restricted room"

        elif old_position in approval_rooms:
            if ask_human(old_position):
                env.apply_action(action)
            else:
                status = "Refused: approval denied"

        else:
            env.apply_action(action)

    else:
        status = "No action"

    action_log.append({
        "position": old_position,
        "action": action,
        "result": status,
        "new_position": env.agent_pos
    })


# RUN THE GUARDED AGENT

env = GridEnvironment(
    rows=4,
    cols=4,
    dirt_probability=0.5,
    seed=11
)

action_log = []

for step in range(200):
    percept = env.get_percept()

    if env.is_clean():
        break

    action = simple_reflex_agent(percept, env)
    guarded_step(env, action)

    assert env.agent_pos not in restricted_rooms, \
        "Restricted room entered!"

print("Steps:", len(action_log))
print("Clean rooms remaining:",
      sum(sum(row) for row in env.dirt))
print("Log entries:", len(action_log))

print("\nLast 10 log entries:")
for entry in action_log[-10:]:
    print(entry)


Steps: 200
Clean rooms remaining: 1
Log entries: 200

Last 10 log entries:
{'position': (2, 1), 'action': 'Down', 'result': 'Allowed', 'new_position': (3, 1)}
{'position': (3, 1), 'action': 'Right', 'result': 'Allowed', 'new_position': (3, 2)}
{'position': (3, 2), 'action': 'Left', 'result': 'Allowed', 'new_position': (3, 1)}
{'position': (3, 1), 'action': 'Left', 'result': 'Allowed', 'new_position': (3, 0)}
{'position': (3, 0), 'action': 'Up', 'result': 'Allowed', 'new_position': (2, 0)}
{'position': (2, 0), 'action': 'Up', 'result': 'Allowed', 'new_position': (1, 0)}
{'position': (1, 0), 'action': 'Down', 'result': 'Allowed', 'new_position': (2, 0)}
{'position': (2, 0), 'action': 'Down', 'result': 'Allowed', 'new_position': (3, 0)}
{'position': (3, 0), 'action': 'Right', 'result': 'Allowed', 'new_position': (3, 1)}
{'position': (3, 1), 'action': 'Left', 'result': 'Allowed', 'new_position': (3, 0)}
